In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [2]:
train_df = pd.read_csv("/content/GSE98320.csv")
val_df   = pd.read_csv("/content/GSE129166.csv")

X_train = train_df.drop(columns=["sample_id", "diagnosis"])
y_train = train_df["diagnosis"]
X_val   = val_df.drop(columns=["sample_id", "diagnosis"])[X_train.columns]
y_val   = val_df["diagnosis"]

In [3]:
# Impute any missing gene values (median fit on training data only)
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val   = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)

class_labels = sorted(y_train.unique())

In [4]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(max_iter=2000, random_state=42)),
])
param_grid = {
    "clf__hidden_layer_sizes": [(64,), (128,), (64, 32)],
    "clf__alpha": [0.0001, 0.001, 0.01],
}

In [5]:
def print_metrics(y_true, y_pred, label):
    print(f"\n=== MLP — {label} ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=class_labels, zero_division=0)
    for cls, pi, ri, fi in zip(class_labels, p, r, f):
        print(f"  {cls:6s} | precision={pi:.4f}  recall={ri:.4f}  f1={fi:.4f}")
    print(f"  MACRO  | precision={np.mean(p):.4f}  recall={np.mean(r):.4f}  f1={np.mean(f):.4f}")

In [6]:
# Nested CV on training set
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
X_train_r, y_train_r = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
oof_pred = np.empty(len(y_train_r), dtype=object)

for fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X_train_r, y_train_r)):
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    search = GridSearchCV(clone(pipeline), param_grid, cv=inner_cv, scoring="accuracy", n_jobs=-1)
    search.fit(X_train_r.iloc[tr_idx], y_train_r.iloc[tr_idx])
    oof_pred[te_idx] = search.predict(X_train_r.iloc[te_idx])
    print(f"  fold {fold+1}/10 done, best params={search.best_params_}")

print_metrics(y_train_r, oof_pred, "Cross-Validation Performance (GSE98320)")

  fold 1/10 done, best params={'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (64, 32)}
  fold 2/10 done, best params={'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (64,)}
  fold 3/10 done, best params={'clf__alpha': 0.01, 'clf__hidden_layer_sizes': (64, 32)}
  fold 4/10 done, best params={'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (128,)}
  fold 5/10 done, best params={'clf__alpha': 0.01, 'clf__hidden_layer_sizes': (64, 32)}
  fold 6/10 done, best params={'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (64, 32)}
  fold 7/10 done, best params={'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (128,)}
  fold 8/10 done, best params={'clf__alpha': 0.01, 'clf__hidden_layer_sizes': (64, 32)}
  fold 9/10 done, best params={'clf__alpha': 0.001, 'clf__hidden_layer_sizes': (64, 32)}
  fold 10/10 done, best params={'clf__alpha': 0.01, 'clf__hidden_layer_sizes': (64, 32)}

=== MLP — Cross-Validation Performance (GSE98320) ===
Accuracy: 0.8857
  ABMR   | precision=0.8419  recall=0.8006 

In [7]:
# Refit on full training set, evaluate on independent validation set (all classes, including TCMR)
final_search = GridSearchCV(clone(pipeline), param_grid,
                             cv=StratifiedKFold(3, shuffle=True, random_state=42),
                             scoring="accuracy", n_jobs=-1)
final_search.fit(X_train, y_train)
val_pred = final_search.predict(X_val)
print_metrics(y_val, val_pred, "Independent Validation Performance (GSE129166)")


=== MLP — Independent Validation Performance (GSE129166) ===
Accuracy: 0.9351
  ABMR   | precision=0.7500  recall=1.0000  f1=0.8571
  NR     | precision=1.0000  recall=0.9167  f1=0.9565
  TCMR   | precision=1.0000  recall=1.0000  f1=1.0000
  MACRO  | precision=0.9167  recall=0.9722  f1=0.9379
